# Retrieval End-to-End Tests

Tests all retrieval paths: BM25 (sparse), dense, hybrid, filtered, thematic (book summaries).
Multi-hop graph retrieval requires Supermemory triples — tested separately once ingestion completes.

**Prerequisites:** Qdrant upload done (`python -m src.config.qdrant_upload`), Vertex AI authenticated, `.env` populated.

In [ ]:
import os
import json
import sys
from pathlib import Path
import numpy as np
import vertexai
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
sys.path.insert(0, str(Path.cwd().parent))
from src.tools.vector_search import vector_search
from src.tools.book_summary_search import book_summary_search
from src.models.agent_contracts import Passage

vertexai.init(project=os.environ["gcp_project"], location=os.environ["gcp_location"])
embedding_model: TextEmbeddingModel = TextEmbeddingModel.from_pretrained("text-embedding-005")
qdrant: QdrantClient = QdrantClient(url=os.environ["qdrant_url"], api_key=os.environ["qdrant_api_key"])
collection: str = "literary_chunks"

with open("../data/book_summaries.json") as f:
    summaries: list[dict] = json.load(f)
summary_embeddings: np.ndarray = np.array([s["embedding"] for s in summaries], dtype=np.float32)

In [ ]:
def show(passages: list[Passage], n: int = 5) -> None:
    """Print passages: score, book, chapter, and a text snippet."""
    for i, p in enumerate(passages[:n]):
        snippet: str = p.text[:120].replace("\n", " ")
        print(f"  [{i+1}] {p.score:.4f}  {p.book_id:<25s}  ch.{p.chapter_number:>3d}  {snippet}...")

## Single-hop factual — BM25 (sparse)
Keyword-heavy query with specific entity names. BM25 should rank exact matches highest.

In [ ]:
q1: str = "Victor Frankenstein creates the creature in his laboratory"
print(f"Query: {q1}")
show(vector_search(q1, "bm25", qdrant, collection, embedding_model))

## Single-hop fuzzy recall — dense
No exact entity names, relies on semantic similarity to find relevant passages.

In [ ]:
q2: str = "a dark ambitious scientist who brings something to life and then regrets it"
print(f"Query: {q2}")
show(vector_search(q2, "dense", qdrant, collection, embedding_model))

## Single-hop mixed — hybrid (RRF)
Mix of entity names and semantic phrasing. Hybrid fuses BM25 + dense via Qdrant RRF.

In [ ]:
q3: str = "Dracula arrives in England aboard the ship Demeter"
print(f"Query: {q3}")
show(vector_search(q3, "hybrid", qdrant, collection, embedding_model))

## Filtered search — scoped to a single book
Same query types but restricted via Qdrant payload filter on `book_id`.

In [ ]:
q4: str = "escape from prison island revenge"
print(f"Query: {q4}  (filtered to count_of_monte_cristo)")
show(vector_search(q4, "hybrid", qdrant, collection, embedding_model, book_id="count_of_monte_cristo"))

print(f"\nQuery: {q4}  (no filter — should surface same book)")
show(vector_search(q4, "hybrid", qdrant, collection, embedding_model))

## Thematic — book summary search
Broad exploratory query over book-level summaries (9 books, in-memory dense cosine).
Returns the most relevant books, not individual passages.

In [ ]:
q5: str = "philosophical exploration of ambition, hubris, and the consequences of playing god"
print(f"Query: {q5}")
results: list[Passage] = book_summary_search(q5, summaries, summary_embeddings, embedding_model, top_k=3)
for i, p in enumerate(results):
    print(f"  [{i+1}] {p.score:.4f}  {p.book_id:<25s}  {p.text[:100].replace(chr(10), ' ')}...")

## Cross-book comparative
Same query run against multiple books to compare retrieval quality across the corpus.

In [ ]:
q6: str = "a journey into a strange and terrifying place"
target_books: list[str] = ["dracula", "alice_in_wonderland", "frankenstein", "beowulf"]

for bid in target_books:
    results = vector_search(q6, "hybrid", qdrant, collection, embedding_model, book_id=bid, top_k=2)
    print(f"{bid}:")
    show(results, n=2)
    print()

## Multi-hop relational (vector fallback)
True multi-hop requires the graph agent traversing Supermemory triples (pending ingestion).
Here we test with hybrid search — useful as a baseline to compare against graph retrieval later.

In [ ]:
q7: str = "Who does Edmond Dantes seek revenge against and what did they do to him?"
print(f"Query: {q7}")
show(vector_search(q7, "hybrid", qdrant, collection, embedding_model, book_id="count_of_monte_cristo"))

q8: str = "What is the relationship between Elizabeth Bennet and Mr Darcy and how does it change?"
print(f"\nQuery: {q8}")
show(vector_search(q8, "hybrid", qdrant, collection, embedding_model, book_id="pride_and_prejudice"))

## Method comparison
Same query across all three methods to observe ranking differences.

In [ ]:
q9: str = "the creature confronts Victor on the glacier"

for method in ("bm25", "dense", "hybrid"):
    print(f"{method.upper()}:")
    show(vector_search(q9, method, qdrant, collection, embedding_model, book_id="frankenstein"), n=3)
    print()